# 01 · Why nesting matters

**The question:** how much does it matter that employees sit inside teams?

Every engagement survey has this shape. People answer about their manager, their workload, and how they feel, and the people who share a team share a lot of their answers. This notebook shows what goes wrong when an analysis ignores that, and how to measure how much it matters.

**The data:** `bh1996`, from Bliese & Halverson (1996). It covers 7,382 U.S. Army soldiers in 99 companies, with ratings of well-being, leadership climate, work hours and cohesion. Read "soldier" as *employee* and "company" as *team*, and you have a Culture Amp dataset.

**By the end you'll be able to:**
1. Compute and explain ICC(1): how much of a variable lives at the team level
2. Explain the design effect: why a small ICC can still wreck your standard errors
3. Show an ordinary regression overstating its certainty, and fix it with a multilevel model
4. Use ICC(2) to judge how many respondents a team score needs
5. Show why "shrunken" team scores beat raw averages for small teams

**How to use this notebook:** when you see **Predict first**, write your guess in the cell before running the next one. The guesses you get wrong are the ones you'll remember in an interview.

In [ ]:
# colab: runtime > change runtime type > R. the first run installs packages (about a minute)
for (pkg in c("multilevel", "ggplot2")) {
  if (!requireNamespace(pkg, quietly = TRUE)) install.packages(pkg)
}
suppressPackageStartupMessages({
  library(multilevel)  # also loads nlme, which fits the models
  library(ggplot2)
})
options(digits = 3, repr.plot.width = 9, repr.plot.height = 4.5)
ctl <- lmeControl(opt = "optim")  # a sturdier optimizer for models with random slopes

# checks your answers in the "your turn" cells
check <- function(yours, target, label = "answer", tol = 0.005) {
  if (is.null(yours)) return(message("[todo] ", label, ": not filled in yet"))
  keep <- !is.na(target)
  gap <- abs(as.numeric(yours)[keep] - as.numeric(target)[keep])
  if (length(yours) == length(target) && !anyNA(gap) && max(gap) <= tol) {
    message("[ok] ", label, ": ", paste(round(head(as.numeric(yours), 3), 3), collapse = ", "))
  } else {
    message("[not yet] ", label, ": yours = ", paste(round(head(as.numeric(yours), 3), 3), collapse = ", "),
            ". recheck the formula")
  }
}

data(bh1996)
head(bh1996)

The `G.` columns are team averages (G for *group*) and the `W.` columns are each soldier's score minus their team's average (W for *within*). The package computed them for you. You'll rebuild them yourself in notebook 02.

## Part A · Look at the structure

**Predict first:** how many soldiers per team, on average? How big is the smallest team?

In [ ]:
teamSizes <- table(bh1996$GRP)
cat("soldiers:", nrow(bh1996), "\nteams:", length(teamSizes), "\n\n")
summary(as.numeric(teamSizes))

ggplot(data.frame(n = as.numeric(teamSizes)), aes(n)) +
  geom_histogram(bins = 30, fill = "grey40") +
  labs(x = "soldiers per team", y = "number of teams", title = "Team sizes")

These teams are big (about 75 people on average), much bigger than the usual 6–12 person manager team in a company. Keep that in mind: team size is about to matter a lot.

## Part B · How much of well-being lives at the team level?

Split each soldier's well-being into a **team part** (their team's average) and an **individual part** (how far they sit from that average). ICC(1) is the share of the total variance that belongs to the team part:

$$\text{ICC(1)} = \frac{\tau_{00}}{\tau_{00} + \sigma^2}$$

where $\tau_{00}$ is the variance *between* team means and $\sigma^2$ is the variance of people *within* teams.

You can estimate it from a one-way ANOVA with team as the factor. With $MS_B$ the between-team mean square, $MS_W$ the within-team mean square, and $k$ the average team size:

$$\text{ICC(1)} = \frac{MS_B - MS_W}{MS_B + (k - 1)\,MS_W}$$

Why this works: $MS_W$ estimates $\sigma^2$. $MS_B$ estimates $\sigma^2 + k\,\tau_{00}$, because a team mean based on $k$ people still carries some individual noise. Solve those two for $\tau_{00}$ and $\sigma^2$, plug them into the first formula, and you get the second.

In [ ]:
wbAov <- aov(WBEING ~ as.factor(GRP), data = bh1996)
summary(wbAov)

In [ ]:
# your turn: compute ICC(1) for well-being by hand
# hint: summary(wbAov)[[1]] is a data frame. its "Mean Sq" column holds MSB (row 1) and MSW (row 2)
# k = average team size = number of soldiers / number of teams
msb <- NULL
msw <- NULL
k <- NULL
myIcc1 <- NULL

check(myIcc1, ICC1(wbAov), "ICC(1) for well-being")

In [ ]:
# the multilevel package runs the same formula. compare a few variables
round(c(
  wellbeing  = ICC1(wbAov),
  leadership = ICC1(aov(LEAD ~ as.factor(GRP), data = bh1996)),
  workHours  = ICC1(aov(HRS ~ as.factor(GRP), data = bh1996)),
  cohesion   = ICC1(aov(COHES ~ as.factor(GRP), data = bh1996))
), 3)

About 4% of the variance in well-being is between teams; the other 96% is between people within the same team. Leadership ratings and work hours are more "shared" (about 13%): soldiers in the same company largely agree on what their leaders are like and work similar hours.

The same number comes out of a **null multilevel model**: a model with no predictors that gives each team its own random intercept:

$$\text{WBEING}_{ij} = \gamma_{00} + u_{0j} + r_{ij}, \qquad u_{0j} \sim N(0, \tau_{00}), \quad r_{ij} \sim N(0, \sigma^2)$$

Soldier $i$ in team $j$ scores the overall mean $\gamma_{00}$, plus their team's bump $u_{0j}$, plus their own deviation $r_{ij}$.

In [ ]:
nullModel <- lme(WBEING ~ 1, random = ~ 1 | GRP, data = bh1996)
VarCorr(nullModel)

varComp <- as.numeric(VarCorr(nullModel)[, "Variance"])
tau00 <- varComp[1]   # between-team variance
sigma2 <- varComp[2]  # within-team variance
cat("\nICC(1) from the null model:", round(tau00 / (tau00 + sigma2), 3))

The ANOVA estimate and the model estimate agree.

**Predict first:** 4% sounds negligible. Suppose you ignore teams and run an ordinary regression with a *team-level* predictor (the team's average leadership rating). Will the standard error be about right, a little too small, or a lot too small? Write a number: "too small by a factor of ___."

## Part C · The design effect

When people in a team resemble each other, each extra person from the same team adds less new information than a person from a new team. The **design effect** measures how much:

$$\text{deff} = 1 + (\bar{k} - 1)\,\text{ICC(1)}$$

- **Effective sample size** = $N / \text{deff}$: how many *independent* people your data is really worth
- **Standard error inflation** ≈ $\sqrt{\text{deff}}$: how much larger the honest standard error is than the one that assumes independence

The thing to notice: deff multiplies the ICC by the **team size**. A tiny ICC times big teams is a big problem.

In [ ]:
icc1 <- ICC1(wbAov)
kBar <- nrow(bh1996) / length(unique(bh1996$GRP))

# your turn: compute the design effect, the effective sample size, and the standard error inflation
designEffect <- NULL
effectiveN <- NULL
seInflation <- NULL

check(designEffect, 1 + (kBar - 1) * icc1, "design effect")
check(effectiveN, nrow(bh1996) / (1 + (kBar - 1) * icc1), "effective n", tol = 1)
check(seInflation, sqrt(1 + (kBar - 1) * icc1), "standard error inflation")

How did your prediction do? About 7,400 soldiers carry roughly the information of 1,800 independent ones, and standard errors that ignore teams are about half the size they should be.

## Part D · Watch it happen

Regress well-being on `G.LEAD`, the team's average leadership rating. That predictor is the same for everyone in a team, so there are really only 99 values of it.

In [ ]:
olsFit <- lm(WBEING ~ G.LEAD, data = bh1996)
mlmFit <- lme(WBEING ~ G.LEAD, random = ~ 1 | GRP, data = bh1996)

olsRow <- coef(summary(olsFit))["G.LEAD", ]
mlmRow <- summary(mlmFit)$tTable["G.LEAD", ]
data.frame(
  model    = c("ordinary regression (ignores teams)", "multilevel (random intercept)"),
  slope    = c(olsRow["Estimate"], mlmRow["Value"]),
  stdError = c(olsRow["Std. Error"], mlmRow["Std.Error"]),
  df       = c(olsFit$df.residual, mlmRow["DF"]),
  row.names = NULL
)

The slopes are nearly the same. The **standard errors** are not: the multilevel one is about 1.6 times larger. Look at the degrees of freedom too. The ordinary regression thinks it has about 7,380 independent pieces of evidence about a team-level relationship. The multilevel model knows it has 99 teams (97 df).

Why 1.6 and not the 2.0 from Part C? Once team leadership is in the model, it explains part of why teams differ in well-being, so the **leftover** (residual) ICC is smaller. The design effect that matters is the one for what the model hasn't explained.

In [ ]:
resVar <- as.numeric(VarCorr(mlmFit)[, "Variance"])
residualIcc <- resVar[1] / sum(resVar)
round(c(
  residualIcc        = residualIcc,
  predictedInflation = sqrt(1 + (kBar - 1) * residualIcc),
  actualInflation    = unname(mlmRow["Std.Error"] / olsRow["Std. Error"])
), 3)

The prediction from the residual ICC lands close to what the models actually did.

**The rule to remember:** ignoring nesting makes standard errors for team-level predictors too small, so you "find" team effects that aren't there. In notebook 03 you'll see a case where ignoring nesting does the *opposite*.

## Part E · How many respondents does a team score need?

This part is the most Culture Amp–specific in the project.

A survey platform reports each team's average. **ICC(2)** is the reliability of that average: how well it would hold up if you re-drew different people from the same team. For a team of $k$ respondents it follows the Spearman–Brown formula:

$$\text{ICC(2)}_k = \frac{k \cdot \text{ICC(1)}}{1 + (k - 1)\,\text{ICC(1)}}$$

A common rule of thumb is .70 or higher before you treat a team mean as a stable property of the team.

**Predict first:** survey platforms often refuse to show results for teams with fewer than 5 respondents, to protect anonymity. For a 5-person team, what's the ICC(2) of its leadership-climate score? Above or below .70?

In [ ]:
iccLead <- ICC1(aov(LEAD ~ as.factor(GRP), data = bh1996))
iccWb <- icc1
spearmanBrown <- function(icc, k) k * icc / (1 + (k - 1) * icc)

teamSize <- c(3, 5, 8, 10, 15, 20, 30, 50)
reliability <- data.frame(
  teamSize,
  leadership = spearmanBrown(iccLead, teamSize),
  wellbeing  = spearmanBrown(iccWb, teamSize)
)
reliability

plotData <- rbind(
  data.frame(teamSize, icc2 = reliability$leadership, measure = "leadership climate"),
  data.frame(teamSize, icc2 = reliability$wellbeing, measure = "well-being")
)
ggplot(plotData, aes(teamSize, icc2, colour = measure)) +
  geom_line() + geom_point() +
  geom_hline(yintercept = 0.7, linetype = "dashed") +
  annotate("text", x = 42, y = 0.74, label = ".70 rule of thumb", size = 3) +
  labs(x = "respondents per team", y = "ICC(2): reliability of the team mean", colour = NULL)

For a 5-person team, the leadership score's reliability is about .42, and the well-being score's is under .20. Leadership needs about 16 respondents to reach .70.

**The lesson:** a minimum reporting size of 5 protects people's **anonymity**. It does not make a 5-person team's score a **reliable** measure of that team. Those are two different thresholds, and a customer looking at a dashboard will treat every team score as if it were reliable.

What can an analyst do? Show uncertainty (intervals) next to small-team scores. Combine survey cycles. Or use the multilevel model's **shrunken** estimates, which is Part F.

## Part F · Shrinkage: better scores for small teams

The multilevel model's estimate of each team's mean is a weighted blend of the team's own raw average and the overall average. Teams with less data get pulled ("shrunk") more toward the middle:

$$\hat\mu_j = \lambda_j\,\bar{y}_j + (1 - \lambda_j)\,\hat\gamma_{00}, \qquad \lambda_j = \frac{\tau_{00}}{\tau_{00} + \sigma^2 / n_j}$$

$\lambda_j$ is that team's ICC(2). An unreliable team mean gets less weight.

**The experiment:** pretend each team had only 5 respondents. Use each team's full roster as the "truth." Which comes closer to the truth: the raw average of 5, or the shrunken estimate?

**Predict first:** by how much will shrinkage cut the error, if at all?

In [ ]:
set.seed(2026)
# pretend each team only had 5 respondents
smallSample <- do.call(rbind, lapply(split(bh1996, bh1996$GRP), function(d) d[sample(nrow(d), 5), ]))

trueMean <- tapply(bh1996$WBEING, bh1996$GRP, mean)          # full team: our stand-in for the truth
rawMean <- tapply(smallSample$WBEING, smallSample$GRP, mean) # what a dashboard would show

smallNull <- lme(WBEING ~ 1, random = ~ 1 | GRP, data = smallSample)
shrunkMean <- setNames(coef(smallNull)[, 1], rownames(coef(smallNull)))

teams <- names(rawMean)
rmse <- function(est) sqrt(mean((est[teams] - trueMean[teams])^2))
round(c(rawError = rmse(rawMean), shrunkError = rmse(shrunkMean)), 3)

In [ ]:
compare <- data.frame(truth = trueMean[teams], raw = rawMean[teams], shrunk = shrunkMean[teams])
ggplot(compare) +
  geom_segment(aes(x = raw, xend = shrunk, y = truth, yend = truth), colour = "grey75") +
  geom_point(aes(raw, truth), colour = "grey40") +
  geom_point(aes(shrunk, truth), colour = "steelblue") +
  geom_abline(linetype = "dashed") +
  labs(x = "estimated team mean   (grey = raw average of 5, blue = shrunken)",
       y = "true team mean (all soldiers)",
       title = "Shrinkage pulls noisy small-team averages toward the middle")

round(c(spreadOfTruth = sd(compare$truth), spreadOfRaw = sd(compare$raw), spreadOfShrunk = sd(compare$shrunk)), 3)

Shrinkage cuts the typical error roughly in half. Look at the spreads, though: the raw averages exaggerate how different teams are, and the shrunken ones understate it.

So:
- To **score or flag one team**, the shrunken estimate is the better bet.
- To **describe how much teams differ**, use $\tau_{00}$ from the model, not the spread of either set of estimates.

## Wrap-up

Write these in your own words in the cell below, as you'd say them to an interviewer. Aim for two sentences each.

1. What ICC(1) is, and why a 4% ICC still mattered here
2. What went wrong with the ordinary regression in Part D, and how the multilevel model fixed it
3. What you'd tell a customer who wants to act on a 5-person team's engagement score

*Your answers:*

1.

2.

3.